# Stage 1 — Reconstructing SegRNN's Table II and Table III
### ETTh1, ETTh2, ETTm1, ETTm2 — multivariate and univariate

## The problem this paper addresses

Long-term time-series forecasting (LTSF) means predicting many steps
into the future — horizons of H=96 up to H=720 timesteps — from a long
history of past observations (here, a look-back window of L=720). This
is hard for two different families of models, for two different
reasons.

**Classical statistical models** (ARIMA, exponential smoothing, and the
other models covered in this course's Time-Series Forecasting lecture)
are built around short-term, largely univariate structure — they assume
relatively simple, stable autocorrelation patterns and don't scale well
to long horizons or many correlated input channels (ETTh1 alone has 7).

**Recurrent neural networks** (RNNs/LSTMs/GRUs) can in principle model
arbitrary long-range dependencies, but in practice degrade badly on long
sequences, for two compounding reasons: (1) processing a 720-step
look-back one timestep at a time means backpropagating through 720
sequential steps, where gradients vanish or explode; and (2) generating
a long forecast one step at a time (autoregressively) means every
prediction feeds into the next one, so small errors compound over the
horizon — by H=720 the model is forecasting from its own accumulated
mistakes. This is why, before this paper, Transformer-based
architectures (Informer, Autoformer, FEDformer, PatchTST, iTransformer)
had become the dominant approach for LTSF: attention looks at the whole
sequence at once, sidestepping both problems — at the cost of being
computationally expensive (attention scales roughly quadratically with
sequence length) and having far more parameters to train.

The paper's question is direct: **can an RNN be redesigned to avoid
both of its usual failure modes, while staying much cheaper than a
Transformer?**

## How SegRNN solves it

SegRNN's answer is two structural changes to a plain GRU, neither of
which adds a new mechanism (no attention, no extra layers) — they
change *what the RNN operates over*:

**1. Segment-wise iteration, not point-wise iteration.** Instead of
feeding the GRU one raw timestep at a time (720 steps for L=720), the
look-back window is first chopped into `n = L/w` non-overlapping
segments of length `w` (24 or 48 here, chosen per dataset), each segment
is linearly embedded into a single d-dimensional vector, and *that*
sequence of `n` segment-vectors (e.g. n=15 or n=30, not 720) is what the
GRU actually encodes. This directly attacks the vanishing-gradient/
long-sequence problem: the GRU only has to propagate information across
a few dozen steps, not hundreds.

**2. Parallel Multi-step Forecasting (PMF), not step-by-step decoding.**
Instead of generating the forecast one segment at a time and feeding
each prediction back in as the next input (the classical,
error-compounding approach — the paper calls this RMF and shows it's
worse), SegRNN generates *all* `m` future segments in a single parallel
pass: each target segment gets its own positional embedding (its
relative position in the forecast horizon, concatenated with which
channel it belongs to), and all `m` of these embeddings are pushed
through the *same* GRU cell — the one already used for encoding — at
once, seeded with the encoder's final hidden state. No prediction ever
feeds into another prediction, so there's no error-accumulation chain
to begin with.

The result, per the paper's own numbers (reproduced below): a
single-layer GRU, doing almost nothing architecturally exotic, matches
or beats Transformer-based SOTA models on most settings, while using a
small fraction of the parameters (the paper's Table VI: >78% less
training time, >82% less peak GPU memory vs. PatchTST). That efficiency
claim is exactly what this project's Stage 2 work has been probing from
many angles (the `d_model` sweep, weight tying, the frozen-reservoir
experiment) — this notebook is the foundation those experiments build
on: reproducing the paper's own accuracy claims, faithfully, before
changing anything.

## What Table II and Table III each check, and why both

**Table II (multivariate)** is the paper's *main* result: all 7 input
channels are used both as input and as forecast target simultaneously,
and SegRNN is compared against 8 baselines across 8 datasets. This is
the setting the paper's headline claims are about, and the one this
project's Stage 2 experiments have all been built on (ETTh1,
multivariate).

**Table III (univariate)** asks a narrower but important question: does
the architecture still hold up in the simpler single-channel setting,
where there's no cross-channel information to exploit? The paper
disables the CP (channel-identity) half of the positional embedding for
this setting, since there's only one channel to identify — everything
else about the architecture is unchanged. Reconstructing both tells us
whether SegRNN's advantage is really about the segment-wise/PMF
mechanism itself (which should hold in both settings) or partly an
artifact of how it handles multiple correlated channels (which would
only show up in Table II).

This notebook reconstructs both tables for the four ETT datasets
(ETTh1, ETTh2, ETTm1, ETTm2) — the two hourly and two 15-minute variants
of the same underlying transformer-load sensor data, now uploaded to
Drive.

## Evaluation protocol (same for every run in this notebook)

- Look-back `L=720`, forecast horizons `H ∈ {96, 192, 336, 720}` — fixed
  by the paper, not tuned per dataset.
- Chronological 6:2:2 train/val/test split (12/4/4 months —
  `data_provider/data_loader.py`'s `Dataset_ETT_hour` for ETTh1/ETTh2,
  `Dataset_ETT_minute` for ETTm1/ETTm2 using the identical border
  formula scaled x4 for 15-minute-frequency data), `StandardScaler` fit
  on the train split only — the same protocol every Stage 2 strand in
  this project has used.
- Metrics: **MSE** and **MAE**, computed on the standardized (scaled)
  values, matching the paper's own evaluation, not inverse-transformed
  back to physical units. One small note on the paper's own tables: the
  second metric column is literally labeled "MSA" in the published PDF
  (`docs/SegRNN_paper.pdf`, Tables II and III), not "MAE" — the values
  match standard MAE exactly (e.g. ETTh1 H=96 multivariate: 0.392,
  consistent with every other citation of this result throughout this
  project), so this reads as a labeling artifact in the published table
  rather than a different metric — worth flagging as a genuine, checkable
  detail rather than glossing over it.
- Every run below uses the *exact* hyperparameters from this repo's own
  `scripts/SegRNN/<dataset>.sh` and `scripts/SegRNN/univariate.sh` —
  these differ meaningfully per dataset (segment length, dropout, batch
  size, learning rate, and whether channel-identity encoding is even
  used). The paper doesn't claim one universal hyperparameter setting;
  it tunes per dataset, and this notebook reproduces that faithfully
  rather than picking one convenient configuration.

**~32 training runs total** (4 datasets x 4 horizons x 2 settings
[multivariate/univariate]) — roughly 2-2.5 hours on a T4, similar per-run
cost to the first Stage 2 notebook. Each dataset/setting combination is
its own Part below, so a Colab disconnect only costs that one part, not
the whole run — restart from Setup and re-run only the Parts you haven't
completed yet.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import os
REPO = "https://github.com/amitzr/SegRNN.git"
if not os.path.exists('/content/proj'):
    !git clone $REPO /content/proj
%cd /content/proj
!git pull

In [ ]:
import os
if not os.path.exists('/content/proj/dataset'):
    os.symlink('/content/drive/MyDrive/ts-project/dataset', '/content/proj/dataset')
!ls -la /content/proj/dataset | head
!pip install -q -r requirements.txt

## Setup: shared constants, training runner, plotting

`run_horizon` here is more general than the one in the Stage 2
notebooks — it's parameterized by dataset name, feature mode
(multivariate `M` / univariate `S`), and every hyperparameter that
varies per dataset, instead of hardcoding ETTh1's own values.

In [ ]:
import os, sys, re, csv, subprocess, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, os.getcwd())  # make scripts.*, utils.*, data_provider.* importable

HORIZONS = [96, 192, 336, 720]
DATASETS = ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2']

# Paper Table II (multivariate), SegRNN column -- read directly off
# docs/SegRNN_paper.pdf's rendered table (page 6), not the paper's own
# text extraction (its table content doesn't extract as plain text).
PAPER_TABLE2 = {
    'ETTh1': {'mse': {96: 0.351, 192: 0.392, 336: 0.423, 720: 0.466},
              'mae': {96: 0.392, 192: 0.414, 336: 0.433, 720: 0.472}},
    'ETTh2': {'mse': {96: 0.276, 192: 0.341, 336: 0.364, 720: 0.403},
              'mae': {96: 0.335, 192: 0.389, 336: 0.403, 720: 0.448}},
    'ETTm1': {'mse': {96: 0.293, 192: 0.328, 336: 0.357, 720: 0.410},
              'mae': {96: 0.354, 192: 0.378, 336: 0.399, 720: 0.430}},
    'ETTm2': {'mse': {96: 0.164, 192: 0.226, 336: 0.284, 720: 0.381},
              'mae': {96: 0.250, 192: 0.294, 336: 0.339, 720: 0.402}},
}

# Paper Table III (univariate), SegRNN column -- same source, page 7.
PAPER_TABLE3 = {
    'ETTh1': {'mse': {96: 0.053, 192: 0.067, 336: 0.079, 720: 0.078},
              'mae': {96: 0.180, 192: 0.208, 336: 0.225, 720: 0.224}},
    'ETTh2': {'mse': {96: 0.125, 192: 0.160, 336: 0.186, 720: 0.209},
              'mae': {96: 0.277, 192: 0.320, 336: 0.350, 720: 0.370}},
    'ETTm1': {'mse': {96: 0.026, 192: 0.040, 336: 0.054, 720: 0.070},
              'mae': {96: 0.122, 192: 0.154, 336: 0.179, 720: 0.202}},
    'ETTm2': {'mse': {96: 0.063, 192: 0.089, 336: 0.117, 720: 0.156},
              'mae': {96: 0.183, 192: 0.224, 336: 0.263, 720: 0.309}},
}

# Exact per-dataset hyperparameters from scripts/SegRNN/<dataset>.sh
MULTIVARIATE_CONFIG = {
    'ETTh1': dict(seg_len=24, enc_in=7, d_model=512, dropout=0.1, channel_id=1, batch_size=64,  learning_rate=0.0003),
    'ETTh2': dict(seg_len=24, enc_in=7, d_model=512, dropout=0.5, channel_id=0, batch_size=64,  learning_rate=0.0003),
    'ETTm1': dict(seg_len=48, enc_in=7, d_model=512, dropout=0.5, channel_id=1, batch_size=256, learning_rate=0.0003),
    'ETTm2': dict(seg_len=48, enc_in=7, d_model=512, dropout=0.5, channel_id=0, batch_size=256, learning_rate=0.0003),
}

# Exact per-dataset hyperparameters from scripts/SegRNN/univariate.sh
UNIVARIATE_CONFIG = {
    'ETTh1': dict(seg_len=48, enc_in=1, d_model=256, dropout=0.5, channel_id=0, batch_size=256, learning_rate=0.0005),
    'ETTh2': dict(seg_len=48, enc_in=1, d_model=256, dropout=0.5, channel_id=0, batch_size=256, learning_rate=0.0005),
    'ETTm1': dict(seg_len=48, enc_in=1, d_model=512, dropout=0.5, channel_id=0, batch_size=256, learning_rate=0.0002),
    'ETTm2': dict(seg_len=48, enc_in=1, d_model=512, dropout=0.5, channel_id=0, batch_size=256, learning_rate=0.0001),
}

os.makedirs('results/figures', exist_ok=True)


def run_horizon(dataset, pred_len, features, seg_len, enc_in, d_model, dropout, channel_id,
                 batch_size, learning_rate, model='SegRNN'):
    """Launch run_longExp.py for one (dataset, horizon, features) combination,
    stream its output live, and parse the final 'mse:X, mae:Y, ms/sample:Z'
    line it prints. Returns (mse, mae, ms_per_sample)."""
    model_id = f'{dataset}_720_{pred_len}_{features}'
    cmd = [
        'python', '-u', 'run_longExp.py',
        '--is_training', '1', '--model_id', model_id, '--model', model, '--data', dataset,
        '--root_path', './dataset/', '--data_path', f'{dataset}.csv',
        '--features', features, '--seq_len', '720', '--pred_len', str(pred_len),
        '--seg_len', str(seg_len), '--enc_in', str(enc_in), '--d_model', str(d_model),
        '--dropout', str(dropout), '--rnn_type', 'gru', '--dec_way', 'pmf',
        '--channel_id', str(channel_id),
        '--train_epochs', '30', '--patience', '5',
        '--itr', '1', '--batch_size', str(batch_size), '--learning_rate', str(learning_rate),
    ]

    print(f'\n{"="*70}\n{dataset}  H={pred_len}  features={features}  '
          f'seg_len={seg_len} d_model={d_model} dropout={dropout} channel_id={channel_id}\n{"="*70}')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in proc.stdout:
        print(line, end='')
        tail.append(line)
        if len(tail) > 5:
            tail.pop(0)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'{dataset} H={pred_len} ({features}) failed (exit {proc.returncode}) -- see output above')

    m = re.search(r'mse:([\d.]+), mae:([\d.]+), ms/sample:([\d.]+)', ''.join(tail))
    if not m:
        raise RuntimeError(f'Could not find mse/mae/ms-per-sample in output for {dataset} H={pred_len} ({features})')
    return float(m.group(1)), float(m.group(2)), float(m.group(3))


# dataviz-validated categorical palette, same as colab_runner.ipynb's own Paper/Reconstruction pair
COLORS = {'Paper': '#2a78d6', 'Reconstruction': '#008300'}
INK_PRIMARY, INK_SECONDARY, INK_MUTED = '#0b0b0b', '#52514e', '#898781'
GRIDLINE, BASELINE_AXIS, SURFACE = '#e1e0d9', '#c3c2b7', '#fcfcfb'


def plot_metric(metric_name, series, title, save_path=None):
    """series: list of (label, {horizon: value}), in display order."""
    n_series = len(series)
    x = np.arange(len(HORIZONS))
    group_width = 0.8
    bar_width = group_width / n_series

    fig, ax = plt.subplots(figsize=(8, 5), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    for i, (label, values) in enumerate(series):
        offsets = x - group_width / 2 + bar_width * (i + 0.5)
        heights = [values[h] for h in HORIZONS]
        bars = ax.bar(offsets, heights, width=bar_width * 0.9, color=COLORS[label],
                       label=label, edgecolor=SURFACE, linewidth=0.5)
        for bar, h in zip(bars, heights):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{h:.3f}',
                     ha='center', va='bottom', fontsize=7.5, color=INK_PRIMARY)

    ax.set_xticks(x)
    ax.set_xticklabels([f'H={h}' for h in HORIZONS], color=INK_SECONDARY)
    ax.set_ylabel(metric_name.upper(), color=INK_SECONDARY)
    ax.set_title(title, color=INK_PRIMARY, fontsize=12, loc='left')
    ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ('top', 'right', 'left'):
        ax.spines[spine].set_visible(False)
    ax.spines['bottom'].set_color(BASELINE_AXIS)
    ax.tick_params(axis='both', which='both', length=0, colors=INK_MUTED)
    ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(0, 1.16),
              ncol=n_series, fontsize=9, labelcolor=INK_SECONDARY)

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200, facecolor=SURFACE)
        print(f'saved {save_path}')
    plt.show()


def make_table(metric_name, series):
    rows = []
    for h in HORIZONS:
        row = {'Horizon': h}
        for label, values in series:
            row[label] = round(values[h], 4)
        rows.append(row)
    print(f'\n{metric_name.upper()}')
    return pd.DataFrame(rows).set_index('Horizon')


def show_comparison(title, results, paper, save_prefix):
    """results: {horizon: (mse, mae, ms)}. paper: {'mse': {...}, 'mae': {...}}."""
    series_mse = [('Paper', paper['mse']), ('Reconstruction', {h: results[h][0] for h in HORIZONS})]
    series_mae = [('Paper', paper['mae']), ('Reconstruction', {h: results[h][1] for h in HORIZONS})]
    display(make_table('mse', series_mse))
    display(make_table('mae', series_mae))
    plot_metric('mse', series_mse, f'{title} -- MSE', save_path=f'results/figures/{save_prefix}_mse.png')
    plot_metric('mae', series_mae, f'{title} -- MAE', save_path=f'results/figures/{save_prefix}_mae.png')

## Part 1 — Table II: multivariate reconstruction

`--features M`, all 7 channels in and out, channel-identity (CP) encoding
enabled where the dataset's own script turns it on. Hyperparameters below
are copied exactly from `scripts/SegRNN/<dataset>.sh` — note they differ
per dataset (segment length, dropout, and whether `channel_id` is even
used), not a single universal setting:

| Dataset | seg_len | d_model | dropout | channel_id | batch_size | lr |
|---|---|---|---|---|---|---|
| ETTh1 | 24 | 512 | 0.1 | 1 | 64 | 3e-4 |
| ETTh2 | 24 | 512 | 0.5 | 0 | 64 | 3e-4 |
| ETTm1 | 48 | 512 | 0.5 | 1 | 256 | 3e-4 |
| ETTm2 | 48 | 512 | 0.5 | 0 | 256 | 3e-4 |

Each dataset is its own Part (1a-1d) so a disconnect only costs that one
dataset's 4 runs, not the whole table.

### Part 1a -- ETTh1 (multivariate)

In [ ]:
table2_ETTh1 = {}
cfg = MULTIVARIATE_CONFIG['ETTh1']
for h in HORIZONS:
    table2_ETTh1[h] = run_horizon('ETTh1', h, features='M', **cfg)

show_comparison('ETTh1 -- Table II (multivariate)', table2_ETTh1, PAPER_TABLE2['ETTh1'], 'table2_ETTh1')

### Part 1b -- ETTh2 (multivariate)

In [ ]:
table2_ETTh2 = {}
cfg = MULTIVARIATE_CONFIG['ETTh2']
for h in HORIZONS:
    table2_ETTh2[h] = run_horizon('ETTh2', h, features='M', **cfg)

show_comparison('ETTh2 -- Table II (multivariate)', table2_ETTh2, PAPER_TABLE2['ETTh2'], 'table2_ETTh2')

### Part 1c -- ETTm1 (multivariate)

In [ ]:
table2_ETTm1 = {}
cfg = MULTIVARIATE_CONFIG['ETTm1']
for h in HORIZONS:
    table2_ETTm1[h] = run_horizon('ETTm1', h, features='M', **cfg)

show_comparison('ETTm1 -- Table II (multivariate)', table2_ETTm1, PAPER_TABLE2['ETTm1'], 'table2_ETTm1')

### Part 1d -- ETTm2 (multivariate)

In [ ]:
table2_ETTm2 = {}
cfg = MULTIVARIATE_CONFIG['ETTm2']
for h in HORIZONS:
    table2_ETTm2[h] = run_horizon('ETTm2', h, features='M', **cfg)

show_comparison('ETTm2 -- Table II (multivariate)', table2_ETTm2, PAPER_TABLE2['ETTm2'], 'table2_ETTm2')

### Part 1 summary -- Table II, all four datasets

In [ ]:
TABLE2_RESULTS = {'ETTh1': table2_ETTh1, 'ETTh2': table2_ETTh2, 'ETTm1': table2_ETTm1, 'ETTm2': table2_ETTm2}

rows = []
for name, results in TABLE2_RESULTS.items():
    for h in HORIZONS:
        mse, mae, ms = results[h]
        p_mse, p_mae = PAPER_TABLE2[name]['mse'][h], PAPER_TABLE2[name]['mae'][h]
        rows.append({
            'Dataset': name, 'Horizon': h,
            'Paper MSE': p_mse, 'Recon MSE': round(mse, 4), 'Delta MSE %': round((mse / p_mse - 1) * 100, 2),
            'Paper MAE': p_mae, 'Recon MAE': round(mae, 4), 'Delta MAE %': round((mae / p_mae - 1) * 100, 2),
        })
table2_summary_df = pd.DataFrame(rows).set_index(['Dataset', 'Horizon'])
display(table2_summary_df)
table2_summary_df.to_csv('results/stage1_table2_summary.csv')
print('saved results/stage1_table2_summary.csv')

## Part 2 — Table III: univariate reconstruction

`--features S`, a single channel (the `OT` target) in and out,
`channel_id=0` for every dataset (per the paper: "the CP encoding module
in SegRNN is disabled for this setting", since there's only one channel
to identify). Hyperparameters below are copied exactly from
`scripts/SegRNN/univariate.sh`:

| Dataset | seg_len | d_model | dropout | batch_size | lr |
|---|---|---|---|---|---|
| ETTh1 | 48 | 256 | 0.5 | 256 | 5e-4 |
| ETTh2 | 48 | 256 | 0.5 | 256 | 5e-4 |
| ETTm1 | 48 | 512 | 0.5 | 256 | 2e-4 |
| ETTm2 | 48 | 512 | 0.5 | 256 | 1e-4 |

Again split into Parts 2a-2d, one per dataset.

### Part 2a -- ETTh1 (univariate)

In [ ]:
table3_ETTh1 = {}
cfg = UNIVARIATE_CONFIG['ETTh1']
for h in HORIZONS:
    table3_ETTh1[h] = run_horizon('ETTh1', h, features='S', **cfg)

show_comparison('ETTh1 -- Table III (univariate)', table3_ETTh1, PAPER_TABLE3['ETTh1'], 'table3_ETTh1')

### Part 2b -- ETTh2 (univariate)

In [ ]:
table3_ETTh2 = {}
cfg = UNIVARIATE_CONFIG['ETTh2']
for h in HORIZONS:
    table3_ETTh2[h] = run_horizon('ETTh2', h, features='S', **cfg)

show_comparison('ETTh2 -- Table III (univariate)', table3_ETTh2, PAPER_TABLE3['ETTh2'], 'table3_ETTh2')

### Part 2c -- ETTm1 (univariate)

In [ ]:
table3_ETTm1 = {}
cfg = UNIVARIATE_CONFIG['ETTm1']
for h in HORIZONS:
    table3_ETTm1[h] = run_horizon('ETTm1', h, features='S', **cfg)

show_comparison('ETTm1 -- Table III (univariate)', table3_ETTm1, PAPER_TABLE3['ETTm1'], 'table3_ETTm1')

### Part 2d -- ETTm2 (univariate)

In [ ]:
table3_ETTm2 = {}
cfg = UNIVARIATE_CONFIG['ETTm2']
for h in HORIZONS:
    table3_ETTm2[h] = run_horizon('ETTm2', h, features='S', **cfg)

show_comparison('ETTm2 -- Table III (univariate)', table3_ETTm2, PAPER_TABLE3['ETTm2'], 'table3_ETTm2')

### Part 2 summary -- Table III, all four datasets

In [ ]:
TABLE3_RESULTS = {'ETTh1': table3_ETTh1, 'ETTh2': table3_ETTh2, 'ETTm1': table3_ETTm1, 'ETTm2': table3_ETTm2}

rows = []
for name, results in TABLE3_RESULTS.items():
    for h in HORIZONS:
        mse, mae, ms = results[h]
        p_mse, p_mae = PAPER_TABLE3[name]['mse'][h], PAPER_TABLE3[name]['mae'][h]
        rows.append({
            'Dataset': name, 'Horizon': h,
            'Paper MSE': p_mse, 'Recon MSE': round(mse, 4), 'Delta MSE %': round((mse / p_mse - 1) * 100, 2),
            'Paper MAE': p_mae, 'Recon MAE': round(mae, 4), 'Delta MAE %': round((mae / p_mae - 1) * 100, 2),
        })
table3_summary_df = pd.DataFrame(rows).set_index(['Dataset', 'Horizon'])
display(table3_summary_df)
table3_summary_df.to_csv('results/stage1_table3_summary.csv')
print('saved results/stage1_table3_summary.csv')

## Part 3 — Analysis

**To be written after Parts 1-2 have real results.** Once both tables
are reconstructed, this section will compare the paper's numbers against
this reconstruction per dataset/horizon/setting, discuss the size and
direction of any gaps, and check whether the paper's own qualitative
claims hold up here — e.g. that SegRNN's advantage grows with horizon,
that the univariate setting is where the gap between SegRNN and the
lightweight DLinear baseline narrows. Left empty on purpose — filling it
in before the numbers exist would mean writing conclusions before
evidence, exactly what this project has avoided everywhere else.

## Optional — save results back into the repo

Appends this notebook's rows to `results/runs.csv`. Commit/push left
commented out on purpose — review `git status`/`git diff` first.

In [ ]:
RUNS_CSV_HEADER = ['run_id','timestamp','model','dataset','horizon','seq_len','seg_len',
                    'd_model','seed','flags','mse','mae','mase','epoch_time_s','params',
                    'peak_mem_mb','notes']
ts = datetime.datetime.now().isoformat(timespec='seconds')
rows = []

for name, results in TABLE2_RESULTS.items():
    cfg = MULTIVARIATE_CONFIG[name]
    for h, (mse, mae, ms) in results.items():
        rows.append([f'SegRNN_{name}_{h}_M_{ts}', ts, 'SegRNN', name, h, 720, cfg['seg_len'], cfg['d_model'], 2024,
                     f"features=M;dropout={cfg['dropout']};channel_id={cfg['channel_id']}",
                     mse, mae, '', '', '', '', 'Stage 1 Table II reconstruction'])

for name, results in TABLE3_RESULTS.items():
    cfg = UNIVARIATE_CONFIG[name]
    for h, (mse, mae, ms) in results.items():
        rows.append([f'SegRNN_{name}_{h}_S_{ts}', ts, 'SegRNN', name, h, 720, cfg['seg_len'], cfg['d_model'], 2024,
                     f"features=S;dropout={cfg['dropout']};channel_id={cfg['channel_id']}",
                     mse, mae, '', '', '', '', 'Stage 1 Table III reconstruction'])

existing = pd.read_csv('results/runs.csv') if os.path.exists('results/runs.csv') else pd.DataFrame(columns=RUNS_CSV_HEADER)
new_df = pd.DataFrame(rows, columns=RUNS_CSV_HEADER)
combined = pd.concat([existing, new_df], ignore_index=True)
combined.to_csv('results/runs.csv', index=False)
print(f'appended {len(rows)} rows to results/runs.csv (total {len(combined)})')

!git add results/runs.csv results/figures/ results/stage1_table2_summary.csv results/stage1_table3_summary.csv
!git status
# review the diff above, then when ready:
# !git commit -m "Stage 1: reconstruct Table II and Table III for ETTh1/ETTh2/ETTm1/ETTm2"
# !git push origin main